
# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [1]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\avigd\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\avigd\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [4]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(10))


,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...
5,"SAN DIEGO, California (CNN) -- More than 100 h...",San Diego mayor declares state of emergency; W...
6,(CNN) -- A former government contract employee...,NEW: Indictment: Man tried to pass nuclear fil...
7,"HELSINKI, Finland (CNN) -- An 18-year-old auth...","NEW: Teen gunman is dead, Finnish police say ...."
8,WASHINGTON (CNN) -- As he awaits a crucial pro...,President Bush to address the Veterans of Fore...
9,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [ ]:

import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
    model.eval()

    summaries = []
    for batch in batch_generator(texts, batch_size):
        inputs = tokenizer(
            ["summarize: " + t for t in batch],
            return_tensors='pt',
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        summaries.extend(decoded)

        del inputs, outputs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return summaries

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = False
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")


c:\Users\avigd\Desktop\GENAI\.venv2\Lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avigd\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


,prompt_text,reference_summary,t5_small_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest..."
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [7]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")


Exact-match accuracy: 0.0000



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [ ]:
# %pip install rouge_score

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=25027 sha256=9af1edd72235d1f51b36bd6eedbfbe84a367488cde167664259d7ac4d444b077
  Stored in directory: c:\users\avigd\appdata\local\pip\cache\wheels\85\9d\af\01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:

import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    return "".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    norm_preds = [normalize_text(p) if p.strip() else p for p in preds]
    norm_refs = [normalize_text(r) if r.strip() else r for r in refs]
    return rouge.compute(predictions=norm_preds, references=norm_refs)

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
print(compute_rouge_score(test_preds, test_refs))


ROUGE sanity check (fill function first):
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.


In [15]:
# 1. Exact match vs. empty prediction
print("Exact match:", compute_rouge_score(["The cat sat on the mat."], ["The cat sat on the mat."]))
print("Empty pred:", compute_rouge_score([""], ["The cat sat on the mat."]))

# 2. Effect of stemming
print("Stemming - exact:", compute_rouge_score(["The dog is running fast."], ["The dog is running fast."]))
print("Stemming - run/running:", compute_rouge_score(["The dog is run fast."], ["The dog is running fast."]))

# 3. N-gram overlap: partial overlap
print("Partial overlap:", compute_rouge_score(
    ["The quick brown fox jumps over the lazy dog."],
    ["The quick brown fox leaps over a sleepy dog."]
))

# 4. Symmetry: swap preds/refs
pred = ["The quick brown fox jumps over the lazy dog and runs away."]
ref = ["The fox jumps."]
print("Original:", compute_rouge_score(pred, ref))
print("Swapped:", compute_rouge_score(ref, pred))

Exact match: {'rouge1': np.float64(1.0), 'rouge2': np.float64(1.0), 'rougeL': np.float64(1.0), 'rougeLsum': np.float64(1.0)}
Empty pred: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}
Stemming - exact: {'rouge1': np.float64(1.0), 'rouge2': np.float64(1.0), 'rougeL': np.float64(1.0), 'rougeLsum': np.float64(1.0)}
Stemming - run/running: {'rouge1': np.float64(0.8000000000000002), 'rouge2': np.float64(0.5), 'rougeL': np.float64(0.8000000000000002), 'rougeLsum': np.float64(0.8000000000000002)}
Partial overlap: {'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.375), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}
Original: {'rouge1': np.float64(0.4), 'rouge2': np.float64(0.15384615384615385), 'rougeL': np.float64(0.4), 'rougeLsum': np.float64(0.4)}
Swapped: {'rouge1': np.float64(0.4), 'rouge2': np.float64(0.15384615384615385), 'rougeL': np.float64(0.4), 'rougeLsum': np.flo


### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [18]:

from transformers import AutoModelForCausalLM, AutoTokenizer

# def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
#     # TODO: implement simple TL;DR generation (careful with max length)
#     raise NotImplementedError("Implement summarize_with_gpt2")

# def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
#     # TODO: for each row, compute ROUGE-L (or ROUGE-1/2) and store results
#     raise NotImplementedError("Implement compute_rouge_per_row")

# RUN_COMPARE = False
# if RUN_COMPARE and 'train_summaries_t5' in locals():
#     # Add other model outputs then compute per-row ROUGE
#     pass

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    # TODO: implement simple TL;DR generation (careful with max length)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.eval()

    summaries = []
    for batch in batch_generator(texts, batch_size):
        prompts = [t + "\nTL;DR:" for t in batch]
        inputs = tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=1024 - max_new_tokens
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id
            )

        input_len = inputs['input_ids'].shape[1]
        generated = outputs[:, input_len:]
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        summaries.extend([d.strip() for d in decoded])

        del inputs, outputs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return summaries


def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    # TODO: for each row, compute ROUGE-L (or ROUGE-1/2) and store results

    rouge1, rouge2, rougeL = [], [], []
    for pred, ref in zip(df[pred_col], df[ref_col]):
        scores = compute_rouge_score([pred], [ref])
        rouge1.append(scores['rouge1'])
        rouge2.append(scores['rouge2'])
        rougeL.append(scores['rougeL'])

    result = df.copy()
    result['rouge1'] = rouge1
    result['rouge2'] = rouge2
    result['rougeL'] = rougeL
    return result

RUN_COMPARE = True
if RUN_COMPARE and 'train_summaries_t5' in locals():
    train_summaries_gpt2 = summarize_with_gpt2(train_df['prompt_text'].tolist(), model_name='gpt2', batch_size=2)

    comparison_df = train_df[['prompt_text', 'prompt_title']].copy()
    comparison_df['t5_small_summary'] = train_summaries_t5
    comparison_df['gpt2_summary'] = train_summaries_gpt2

    comparison_df = compute_rouge_per_row(comparison_df, pred_col='t5_small_summary')
    comparison_df = comparison_df.rename(columns={'rouge1': 't5_rouge1', 'rouge2': 't5_rouge2', 'rougeL': 't5_rougeL'})

    comparison_df = compute_rouge_per_row(comparison_df, pred_col='gpt2_summary')
    comparison_df = comparison_df.rename(columns={'rouge1': 'gpt2_rouge1', 'rouge2': 'gpt2_rouge2', 'rougeL': 'gpt2_rougeL'})

    display(comparison_df.head())

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

,prompt_text,prompt_title,t5_small_summary,gpt2_summary,t5_rouge1,t5_rouge2,t5_rougeL,gpt2_rouge1,gpt2_rouge2,gpt2_rougeL
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,The race was a bit of a mess for Lewis Hamilto...,0.222222,0.131148,0.222222,0.179104,0.092308,0.179104
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,"The Aqua Dots toys are a ""date rape"" drug that...",0.147059,0.000000,0.117647,0.216216,0.055556,0.108108
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...,The Qualcomm Corporation is a global leader in...,0.424242,0.125000,0.333333,0.264706,0.030303,0.205882
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...",ongoing violence in the country. The governmen...,0.203390,0.035088,0.203390,0.136986,0.028169,0.082192
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,The WTA Tour is a great way to get back into t...,0.379310,0.142857,0.241379,0.123077,0.063492,0.123077



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [19]:

import pandas as pd

def compare_models(rouge_dict):
    # TODO: take {model_name: rouge_scores_dict} -> DataFrame with averages
    return pd.DataFrame(rouge_dict).T

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    # TODO: subset columns for side-by-side viewing
    base_cols = ['prompt_text', 'prompt_title']
    return df[base_cols + pred_cols]


In [20]:
rouge_dict = {
    't5_small': compute_rouge_score(train_summaries_t5, train_df['prompt_title'].tolist()),
    'gpt2': compute_rouge_score(train_summaries_gpt2, train_df['prompt_title'].tolist()),
}
display(compare_models(rouge_dict))
display(compare_models_summaries(comparison_df, ['t5_small_summary', 'gpt2_summary']))

,rouge1,rouge2,rougeL,rougeLsum
t5_small,0.262524,0.092623,0.202341,0.204703
gpt2,0.173228,0.037134,0.132377,0.137483


,prompt_text,prompt_title,t5_small_summary,gpt2_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,The race was a bit of a mess for Lewis Hamilto...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,"The Aqua Dots toys are a ""date rape"" drug that..."
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...,The Qualcomm Corporation is a global leader in...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...",ongoing violence in the country. The governmen...
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,The WTA Tour is a great way to get back into t...
...,...,...,...,...
95,"PARIS, France -- World number three Novak Djok...",World No. 3 Novak Djokovic beaten in the secon...,world number three Novak Djokovic crashes out ...,The first round of the Paris Masters is set fo...
96,(CNN) -- A girl who was shown on a videotape b...,"Stiles described as ""survivalist type"" who car...","new: police say girl found safe, safe, safe. n...",The girl was sexually assaulted in Las Vegas. ...
97,"SAN FRANCISCO, California (CNN) -- A magnitud...","2,000 customers without electricity, power com...","quake left about 2,000 customers without power...",The quake struck at 4:42 a.m. PT (7:42 a.m. ET...
98,WASHINGTON (CNN) -- U.S. first lady Laura Bush...,"Laura Bush calls on Myanmar junta to ""step asi...","first lady Laura Bush calls on junta to ""step ...",The U.S. State Department has been working on ...



## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.


ROUGE-1 and ROUGE-L felt the most informative: ROUGE-1 reflects shared vocabulary or content, while ROUGE-L tolerates some reordering while still capturing structure. ROUGE-2 was the least forgiving, it dropped sharply with even minor rewording, since it requires exact consecutive word matches.

Model size didn’t translate directly into higher ROUGE scores. Larger models produced more fluent, coherent summaries qualitatively, but ROUGE only measures lexical overlap with the reference, so a well-written but differently-worded summary can score lower than a clumsy one that happens to reuse the reference’s exact words.

ROUGE breaks down whenever the correct meaning is expressed with different words: paraphrasing, synonyms, or valid summaries of different length/style than the reference all get penalized despite being semantically correct.

To extend this, human evaluation would catch quality differences ROUGE misses, and semantic-embedding metrics would better reward paraphrases. Adversarial probes, injecting contradictions, negations, or unsupported facts into the source, would test whether models hallucinate or stay faithful to the input rather than just matching reference wording.